In [1]:
import pandas as pd

In [2]:
def rename_columns(df, old_cols, new_cols):
    """
    Rename a series of columns using two aligned lists.
    """
    if len(old_cols) != len(new_cols):
        raise ValueError("old_cols and new_cols must have the same length")

    return df.rename(columns=dict(zip(old_cols, new_cols)))

In [75]:
file_loc = r"D BRKR 1-N Under Const 2026.xlsx"
df = pd.read_excel(file_loc, sheet_name = "Distribution Breakers", index_col=None, na_values=['NA'], skiprows=0, engine='openpyxl')

sel_cols = ['Equipment', 'Substation Name',
            'Reliability Risk Matrix (PoF,CoF)']

df_d_brkr = df[sel_cols]

In [77]:
len(df_d_brkr)

3014

In [78]:
len(df_d_brkr.drop_duplicates(subset=['Equipment']))

3014

In [79]:
def get_first_digit(val):
    try:
        if pd.isna(val):  # Handle NaN
            return None
        val_str = str(abs(int(val)))  # Convert to string, remove sign
        return int(val_str[0])  # First character as integer
    except (ValueError, TypeError):
        return None

In [80]:
df_d_brkr['R'] =df_d_brkr['Reliability Risk Matrix (PoF,CoF)'].apply(get_first_digit)

C:\Users\SIH5\AppData\Local\Temp\ipykernel_25800\1157495278.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_d_brkr['R'] =df_d_brkr['Reliability Risk Matrix (PoF,CoF)'].apply(get_first_digit)


In [81]:
df_d_brkr

,Equipment,Substation Name,"Reliability Risk Matrix (PoF,CoF)",R
0,42757556,PAUL SWEET SUB,23,2
1,43163754,SWIFT SUB,23,2
2,42757558,PAUL SWEET SUB,23,2
3,44641543,EDENVALE SUB,13,1
4,42604437,HICKS SUB,23,2
...,...,...,...,...
3009,40997252,SAN LEANDRO U SUB,53,5
3010,40995446,BAHIA SUB,43,4
3011,41335751,NORIEGA SUB,22,2
3012,40995357,SAN FRAN P(HUNTERS POINT) SUB,43,4


In [82]:
dict_ = {'1': None, '2': None, '3': None, '4': None, '5': None}
for i, j in dict_.items():
    dict_[i] = len(df_d_brkr[df_d_brkr['R'] == int(i)])

In [83]:
dict_

{'1': 225, '2': 1070, '3': 460, '4': 1059, '5': 200}

## Failures

In [105]:
df = pd.read_excel('2024 + Emergency Job Tracker_5_7_25.xlsm', sheet_name = "Emerg Tracker (TCR) 2024 +", index_col=None, na_values=['NA'], skiprows=3, engine='openpyxl')
sel_cols = ['SAP Equipment Number(s), if available', 'Project Name for SAP (Failed Asset)\n(max. 40 char.)','Equipment Type\n(being replaced)',
        'Voltage (kV)', 'Location\n(Station Name)','Future\nYear\nImpact\n($000)', 
        'Estimated Spend in Current Year\n($000)','Date SAM first contacted',
        'Date Declared Emergency (Date decision to be placed in emergency)',
       'Failure Type', 'In-Serv./Catastrophic Equipment Category (CB, Xfmr, Unitsub, 3ph Reg, Battery)',
       'Outage Customer Minutes (CMIN) In-Service and Catastrophic Failures (XFMR, CB, XFMR Bushings, Minor)',
       'Age at Year of Failure', 'MVA/KVA (Transformer & Regs)', '3Ph or 1Ph? (Transformers and Regs)']

C:\Users\SIH5\Anaconda3\envs\sih5\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [106]:
df_fail = df[sel_cols]

df_fail = rename_columns(df_fail, sel_cols, ["sap_id", "Project Name", "Equipment Type", "Voltage", "Substation Name", 
                                            "Future Spending", "Current Spending","Discover Date", "Date", "Failure Type",
                                            "Catastrophic Equipment", "CMIN", "Age", "MVA/KVA", "3Ph or 1Ph"])

In [107]:
df_fail['Date'] = pd.to_datetime(df_fail['Discover Date'], errors='coerce')
df_fail['Year'] = df_fail['Date'].dt.year
df_fail['Month'] = df_fail['Date'].dt.month

In [118]:
df_fail = df_fail[~df_fail["Year"].isna()]

In [119]:
df_txfr_2025 =df_fail[(df_fail["Equipment Type"]=="TXFR") & (df_fail['Date']>=f'01-01-2025') & (df_fail['Date']<f'01-01-2026')]
t_count = df_txfr_2025.shape[0]


df_txfr_2025 = df_txfr_2025[["sap_id", "Equipment Type", "Voltage", "Substation Name", 
                                        "Future Spending", "Current Spending", "Date", "Age", "Failure Type"]]


df_txfr_2025 = rename_columns(df_txfr_2025, ['sap_id'], ['Equipment'])

#######################################################################################################################
df_txfr_2026 =df_fail[(df_fail["Equipment Type"]=="TXFR") & (df_fail['Date']>=f'01-01-2026') & (df_fail['Date']<f'01-01-2027')]
t_count = df_txfr_2026.shape[0]


df_txfr_2026 = df_txfr_2026[["sap_id", "Equipment Type", "Voltage", "Substation Name", 
                                        "Future Spending", "Current Spending", "Date", "Age", "Failure Type"]]


df_txfr_2026 = rename_columns(df_txfr_2026, ['sap_id'], ['Equipment'])

#######################################################################################################################
df_brkr_2025 =df_fail[(df_fail["Equipment Type"]=="BRKR") & (df_fail['Date']>=f'01-01-2025') & (df_fail['Date']<f'01-01-2026')]
t_count = df_brkr_2025.shape[0]


df_brkr_2025 = df_brkr_2025[["sap_id", "Equipment Type", "Voltage", "Substation Name", 
                                        "Future Spending", "Current Spending", "Date", "Age", "Failure Type"]]


df_brkr_2025 = rename_columns(df_brkr_2025, ['sap_id'], ['Equipment'])

#######################################################################################################################
df_brkr_2026 =df_fail[(df_fail["Equipment Type"]=="BRKR") & (df_fail['Date']>=f'01-01-2026') & (df_fail['Date']<f'01-01-2027')]
t_count = df_brkr_2026.shape[0]


df_brkr_2026 = df_brkr_2026[["sap_id", "Equipment Type", "Voltage", "Substation Name", 
                                        "Future Spending", "Current Spending", "Date", "Age", "Failure Type"]]


df_brkr_2026 = rename_columns(df_brkr_2026, ['sap_id'], ['Equipment'])

In [117]:
print(len(df_d_brkr), len(df_t_brkr), len(df_t_txfr), len(df_d_txfr))

3014 3734 456 1923


In [120]:
print(len(df_txfr_2025), len(df_txfr_2026), len(df_brkr_2025), len(df_brkr_2026))

84 8 102 21


In [151]:
pd.merge(df_d_brkr, df_brkr_2026, on='Equipment', how='inner')

,Equipment,Substation Name_x,"Reliability Risk Matrix (PoF,CoF)",R,Equipment Type,Voltage,Substation Name_y,Future Spending,Current Spending,Date,Age,Failure Type
0,40990010,WOODLAND SUB,43,4,BRKR,12,Woodland,NaN,450,2026-03-30,NaN,Catastrophic
1,40992541,HARTER SUB,42,4,BRKR,59E,Harter,NaN,600,2026-04-23,NaN,JIT
2,41180579,WELLFIELD SUB,32,3,BRKR,12,Wellfield,NaN,400,2026-01-31,NaN,JIT
3,42627555,CLOVIS SUB,23,2,BRKR,21,Clovis,NaN,200,2026-02-04,NaN,JIT


In [124]:
asset_files_brkr = pd.concat([df_d_brkr, df_t_brkr])
len(asset_files_brkr.drop_duplicates(subset='Equipment'))

6747

In [125]:
asset_files_txfr = pd.concat([df_d_txfr, df_t_txfr])
len(asset_files_txfr.drop_duplicates(subset='Equipment'))

2373

In [126]:
print(len(asset_files_brkr), len(asset_files_txfr))

6748 2379


In [135]:
duplicates_name = asset_files_brkr[asset_files_brkr.duplicated(subset=['Equipment'])]
print("\nDuplicate rows based on 'Name':\n", duplicates_name)


Duplicate rows based on 'Name':
      Equipment Substation Name  Reliability Risk Matrix (PoF,CoF)  R
660   40991421             NaN                                 33  3


In [136]:
duplicates_name = asset_files_txfr[asset_files_txfr.duplicated(subset=['Equipment'])]
print("\nDuplicate rows based on 'Name':\n", duplicates_name)


Duplicate rows based on 'Name':
      Equipment Substation Name  Reliability Risk Matrix (PoF,CoF)  R
149   41158251         HAAS PH                                 44  4
150   41158253         HAAS PH                                 44  4
151   41158254         HAAS PH                                 44  4
323   44918680   PIT #1 PH SUB                                 13  3
324   45274594   PIT #1 PH SUB                                 13  3
325   45247388   PIT #1 PH SUB                                 23  3
